# Personalized Item Recommendation (PIR) - Repeat-Aware LightGBM Ranker

Notebook này cải thiện ranker purchase/event bằng các feature đo khả năng item được mua lặp lại qua tháng:

- Candidate set = item user đã mua trước đó + item user đã `view_item`/`add_to_cart`.
- Label validation = purchase trong tháng kế tiếp.
- Cải thiện chính = thêm repeat-rate item theo `previous month -> target month` và `history -> target month`, sau đó kiểm chứng trực tiếp với `precision@10`, IoU, MRR và MAP@10 trên December validation.


In [ ]:
import os
from pathlib import Path

print("Current working directory:", os.getcwd())
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
    drive.mount('/content/drive')
except Exception:
    IN_COLAB = False
    print("Running locally; Google Drive mount skipped.")


In [ ]:
from __future__ import annotations

import gc
import json
import pickle
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import pyarrow.parquet as pq
import seaborn as sns
from IPython.display import display
from lightgbm import LGBMClassifier

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "raw_data" / "transaction_full_2025.parquet").exists():
            return candidate
    raise FileNotFoundError("Cannot find project root containing raw_data/transaction_full_2025.parquet")


if IN_COLAB:
    PROJECT_ROOT = Path("/content/drive/MyDrive/ColabData")
    RAW_DIR = PROJECT_ROOT / "lzu"
    NOTEBOOK_DIR = PROJECT_ROOT / "PIR"
else:
    PROJECT_ROOT = find_project_root(Path.cwd().resolve())
    RAW_DIR = PROJECT_ROOT / "raw_data"
    NOTEBOOK_DIR = PROJECT_ROOT / "Additional Doc" / "Personalized Item Recommendation (PIR)"

OUTPUT_PATH = NOTEBOOK_DIR / "submission.pkl"
BASE_REPEAT_SUBMISSION_PATH = NOTEBOOK_DIR / "submission_all_items_repeat_features.pkl"
VALIDATION_SUMMARY_PATH = NOTEBOOK_DIR / "pir_notebook_improvement_summary.json"
FINAL_SEGMENT_BRAND_SUMMARY_PATH = NOTEBOOK_DIR / "pir_final_segment_brand_summary.json"

TRANSACTION_PATH = RAW_DIR / "transaction_full_2025.parquet"
EVENT_PATH = RAW_DIR / "event_full_2025.parquet"
ITEM_PATH = RAW_DIR / "items.parquet"

TRAIN_START = datetime(2025, 1, 1)
SEP_2025 = datetime(2025, 9, 1)
OCT_2025 = datetime(2025, 10, 1)
NOV_2025 = datetime(2025, 11, 1)
DEC_2025 = datetime(2025, 12, 1)
JAN_2026 = datetime(2026, 1, 1)
K = 10
SEGMENT_WIDTH = 500_000

for path in [TRANSACTION_PATH, EVENT_PATH, ITEM_PATH]:
    assert path.exists(), f"Missing file: {path}"

print("Project root:", PROJECT_ROOT)
print("Base repeat-aware:", BASE_REPEAT_SUBMISSION_PATH)
print("Final output:", OUTPUT_PATH)


## 1. Audit dữ liệu

Cell này xác nhận schema, khoảng ngày và phân phối tháng. Điểm quan trọng: `transaction_full_2025.parquet` chỉ có purchase năm 2025, không có blind `01/2026`.

In [ ]:
def parquet_overview(path: Path) -> dict:
    pf = pq.ParquetFile(path)
    return {
        "file": path.name,
        "rows": pf.metadata.num_rows,
        "row_groups": pf.num_row_groups,
        "columns": len(pf.schema_arrow.names),
    }


display(pd.DataFrame([parquet_overview(p) for p in [TRANSACTION_PATH, EVENT_PATH, ITEM_PATH]]))

txn_summary = (
    pl.scan_parquet(TRANSACTION_PATH)
    .select(
        pl.len().alias("rows"),
        pl.col("customer_id").n_unique().alias("customers"),
        pl.col("item_id").n_unique().alias("items"),
        pl.col("updated_date").min().alias("min_date"),
        pl.col("updated_date").max().alias("max_date"),
    )
    .collect()
)
event_summary = (
    pl.scan_parquet(EVENT_PATH)
    .select(
        pl.len().alias("rows"),
        pl.col("customer_id").n_unique().alias("customers"),
        pl.col("item_id").n_unique().alias("items"),
        pl.col("event_date").min().alias("min_date"),
        pl.col("event_date").max().alias("max_date"),
    )
    .collect()
)

print("Transactions")
display(txn_summary)
print("Behavior events")
display(event_summary)

txn_monthly = (
    pl.scan_parquet(TRANSACTION_PATH)
    .with_columns(pl.col("updated_date").dt.strftime("%Y-%m").alias("month"))
    .group_by("month")
    .agg(
        pl.len().alias("purchase_rows"),
        pl.col("customer_id").n_unique().alias("customers"),
        pl.col("item_id").n_unique().alias("items"),
    )
    .sort("month")
    .collect()
)
event_monthly = (
    pl.scan_parquet(EVENT_PATH)
    .with_columns(pl.col("event_date").dt.strftime("%Y-%m").alias("month"))
    .group_by("month")
    .agg(
        pl.len().alias("event_rows"),
        pl.col("customer_id").n_unique().alias("customers"),
        pl.col("item_id").n_unique().alias("items"),
    )
    .sort("month")
    .collect()
)
display(txn_monthly)
display(event_monthly.tail(12))

fig, axes = plt.subplots(1, 2, figsize=(15, 4), constrained_layout=True)
sns.barplot(data=txn_monthly.to_pandas(), x="month", y="purchase_rows", ax=axes[0], color="#2563eb")
axes[0].set_title("Purchase rows by month")
axes[0].tick_params(axis="x", rotation=55)
sns.barplot(data=event_monthly.to_pandas(), x="month", y="event_rows", ax=axes[1], color="#f97316")
axes[1].set_title("Behavior rows by month")
axes[1].tick_params(axis="x", rotation=55)
plt.show()

## 2. Feature Builder

Mỗi dòng training là một cặp `(customer_id, item_id)` candidate. Candidate được lấy từ union:

- user-item đã từng purchase trong history.
- user-item đã từng `view_item` hoặc `add_to_cart` trong history.

Nhóm feature mới ở bản này đo độ lặp lại của từng item:

- `item_repeat_rate_m1`: tỷ lệ user mua item ở tháng trước và tiếp tục mua ở tháng target.
- `item_hist_to_last_rate`: tỷ lệ user từng mua item trong lịch sử và quay lại mua ở tháng target.

Hai tỷ lệ được smoothing để tránh item ít dữ liệu làm nhiễu model.


In [ ]:
BASE_FEATURE_COLS = [
    "qty_total", "cnt_total", "last_month", "active_months",
    "qty_m1", "qty_m2", "qty_m3", "qty_m6", "cnt_m1", "cnt_m2",
    "hl1", "hl2",
    "event_score_total", "atc_count", "view_count", "last_event_month",
    "event_score_m1", "event_hl1",
    "item_qty_total", "item_users_total", "item_qty_m1", "item_users_m1",
    "user_qty_total", "user_unique_items", "user_rows",
    "days_since_last", "days_since_first", "days_since_event",
    "avg_qty_per_row", "recent_qty_share", "item_recent_share",
    "has_purchase_history", "has_event_history",
]

REPEAT_FEATURE_COLS = [
    "item_prev_users", "item_last_users", "item_repeat_rate_m1",
    "item_hist_users", "item_hist_to_last_rate",
]

FEATURE_COLS = BASE_FEATURE_COLS + REPEAT_FEATURE_COLS


def build_item_repeat_features(hist: pl.LazyFrame, hist_last_month: int) -> pl.LazyFrame:
    prev_month = hist_last_month - 1
    last_month = hist_last_month

    prev = hist.filter(pl.col("month") == prev_month).select("customer_id", "item_id").unique()
    last = hist.filter(pl.col("month") == last_month).select("customer_id", "item_id").unique()
    hist_before_last = hist.filter(pl.col("month") < last_month).select("customer_id", "item_id").unique()

    prev_counts = prev.group_by("item_id").agg(pl.len().alias("item_prev_users"))
    last_counts = last.group_by("item_id").agg(pl.len().alias("item_last_users"))
    repeat_counts = (
        prev.join(last, on=["customer_id", "item_id"], how="inner")
        .group_by("item_id")
        .agg(pl.len().alias("item_repeat_users_m1"))
    )
    hist_counts = hist_before_last.group_by("item_id").agg(pl.len().alias("item_hist_users"))
    hist_repeat_counts = (
        hist_before_last.join(last, on=["customer_id", "item_id"], how="inner")
        .group_by("item_id")
        .agg(pl.len().alias("item_hist_to_last_users"))
    )

    base = (
        prev_counts
        .join(last_counts, on="item_id", how="full", coalesce=True)
        .join(repeat_counts, on="item_id", how="left")
        .join(hist_counts, on="item_id", how="full", coalesce=True)
        .join(hist_repeat_counts, on="item_id", how="left")
        .with_columns(
            pl.col("item_prev_users").fill_null(0),
            pl.col("item_last_users").fill_null(0),
            pl.col("item_repeat_users_m1").fill_null(0),
            pl.col("item_hist_users").fill_null(0),
            pl.col("item_hist_to_last_users").fill_null(0),
        )
    )

    return base.with_columns(
        ((pl.col("item_repeat_users_m1") + 5.0) / (pl.col("item_prev_users") + 50.0)).cast(pl.Float32).alias("item_repeat_rate_m1"),
        ((pl.col("item_hist_to_last_users") + 10.0) / (pl.col("item_hist_users") + 100.0)).cast(pl.Float32).alias("item_hist_to_last_rate"),
        pl.col("item_prev_users").cast(pl.Float32),
        pl.col("item_last_users").cast(pl.Float32),
        pl.col("item_hist_users").cast(pl.Float32),
    ).select(
        "item_id",
        "item_prev_users",
        "item_last_users",
        "item_repeat_rate_m1",
        "item_hist_users",
        "item_hist_to_last_rate",
    )


def build_union_features(
    cutoff_dt: datetime,
    hist_last_month: int,
    *,
    target_start: datetime | None = None,
    target_end: datetime | None = None,
    sample_neg_pct: int | None = None,
    hash_seed: int = 987,
) -> pl.DataFrame:
    t0 = time.time()
    print(f"Build features up to {cutoff_dt:%Y-%m-%d} ...")

    hist = (
        pl.scan_parquet(TRANSACTION_PATH)
        .filter((pl.col("updated_date") >= TRAIN_START) & (pl.col("updated_date") < cutoff_dt))
        .select("customer_id", "item_id", "quantity", "updated_date")
        .with_columns(pl.col("updated_date").dt.month().alias("month"))
    )

    purchase_features = hist.group_by(["customer_id", "item_id"]).agg(
        pl.col("quantity").sum().cast(pl.Float32).alias("qty_total"),
        pl.len().cast(pl.Float32).alias("cnt_total"),
        pl.col("updated_date").max().alias("last_date"),
        pl.col("updated_date").min().alias("first_date"),
        pl.col("month").max().cast(pl.Int16).alias("last_month"),
        pl.col("month").n_unique().cast(pl.Float32).alias("active_months"),
        pl.when(pl.col("month") >= hist_last_month).then(pl.col("quantity")).otherwise(0).sum().cast(pl.Float32).alias("qty_m1"),
        pl.when(pl.col("month") >= hist_last_month - 1).then(pl.col("quantity")).otherwise(0).sum().cast(pl.Float32).alias("qty_m2"),
        pl.when(pl.col("month") >= hist_last_month - 2).then(pl.col("quantity")).otherwise(0).sum().cast(pl.Float32).alias("qty_m3"),
        pl.when(pl.col("month") >= hist_last_month - 5).then(pl.col("quantity")).otherwise(0).sum().cast(pl.Float32).alias("qty_m6"),
        pl.when(pl.col("month") >= hist_last_month).then(1).otherwise(0).sum().cast(pl.Float32).alias("cnt_m1"),
        pl.when(pl.col("month") >= hist_last_month - 1).then(1).otherwise(0).sum().cast(pl.Float32).alias("cnt_m2"),
        (pl.col("quantity") * (0.5 ** (hist_last_month - pl.col("month")))).sum().cast(pl.Float32).alias("hl1"),
        (pl.col("quantity") * (0.5 ** ((hist_last_month - pl.col("month")) / 2.0))).sum().cast(pl.Float32).alias("hl2"),
    )

    events = (
        pl.scan_parquet(EVENT_PATH)
        .filter((pl.col("event_date") >= TRAIN_START) & (pl.col("event_date") < cutoff_dt))
        .select("customer_id", "item_id", "event_type", "event_date")
        .with_columns(
            pl.col("event_date").dt.month().alias("event_month"),
            pl.when(pl.col("event_type") == "add_to_cart").then(3.0).otherwise(1.0).alias("event_w"),
            (pl.col("event_type") == "add_to_cart").cast(pl.Float32).alias("is_atc"),
            (pl.col("event_type") == "view_item").cast(pl.Float32).alias("is_view"),
        )
    )

    event_features = events.group_by(["customer_id", "item_id"]).agg(
        pl.col("event_w").sum().cast(pl.Float32).alias("event_score_total"),
        pl.col("is_atc").sum().cast(pl.Float32).alias("atc_count"),
        pl.col("is_view").sum().cast(pl.Float32).alias("view_count"),
        pl.col("event_date").max().alias("last_event_date"),
        pl.col("event_month").max().cast(pl.Int16).alias("last_event_month"),
        pl.when(pl.col("event_month") >= hist_last_month).then(pl.col("event_w")).otherwise(0).sum().cast(pl.Float32).alias("event_score_m1"),
        (pl.col("event_w") * (0.5 ** (hist_last_month - pl.col("event_month")))).sum().cast(pl.Float32).alias("event_hl1"),
    )

    item_stats = hist.group_by("item_id").agg(
        pl.col("quantity").sum().cast(pl.Float32).alias("item_qty_total"),
        pl.col("customer_id").n_unique().cast(pl.Float32).alias("item_users_total"),
        pl.when(pl.col("month") >= hist_last_month).then(pl.col("quantity")).otherwise(0).sum().cast(pl.Float32).alias("item_qty_m1"),
        pl.when(pl.col("month") >= hist_last_month).then(pl.col("customer_id")).otherwise(None).n_unique().cast(pl.Float32).alias("item_users_m1"),
    )

    user_stats = hist.group_by("customer_id").agg(
        pl.col("quantity").sum().cast(pl.Float32).alias("user_qty_total"),
        pl.col("item_id").n_unique().cast(pl.Float32).alias("user_unique_items"),
        pl.len().cast(pl.Float32).alias("user_rows"),
    )

    repeat_features = build_item_repeat_features(hist, hist_last_month)

    features = (
        purchase_features
        .join(event_features, on=["customer_id", "item_id"], how="full", coalesce=True)
        .join(item_stats, on="item_id", how="left")
        .join(user_stats, on="customer_id", how="left")
        .join(repeat_features, on="item_id", how="left")
    )

    fill_zero_cols = [
        "qty_total", "cnt_total", "active_months", "qty_m1", "qty_m2", "qty_m3", "qty_m6",
        "cnt_m1", "cnt_m2", "hl1", "hl2",
        "event_score_total", "atc_count", "view_count", "event_score_m1", "event_hl1",
        "item_qty_total", "item_users_total", "item_qty_m1", "item_users_m1",
        "user_qty_total", "user_unique_items", "user_rows",
        "item_prev_users", "item_last_users", "item_repeat_rate_m1", "item_hist_users", "item_hist_to_last_rate",
    ]

    features = features.with_columns([pl.col(c).fill_null(0) for c in fill_zero_cols])
    features = features.with_columns(
        pl.col("last_month").fill_null(0),
        pl.col("last_event_month").fill_null(0),
        pl.when(pl.col("last_date").is_not_null()).then((pl.lit(cutoff_dt) - pl.col("last_date")).dt.total_days()).otherwise(999).cast(pl.Float32).alias("days_since_last"),
        pl.when(pl.col("first_date").is_not_null()).then((pl.lit(cutoff_dt) - pl.col("first_date")).dt.total_days()).otherwise(999).cast(pl.Float32).alias("days_since_first"),
        pl.when(pl.col("last_event_date").is_not_null()).then((pl.lit(cutoff_dt) - pl.col("last_event_date")).dt.total_days()).otherwise(999).cast(pl.Float32).alias("days_since_event"),
        (pl.col("qty_total") / (pl.col("cnt_total") + 1e-3)).cast(pl.Float32).alias("avg_qty_per_row"),
        (pl.col("qty_m1") / (pl.col("qty_total") + 1e-3)).cast(pl.Float32).alias("recent_qty_share"),
        (pl.col("item_qty_m1") / (pl.col("item_qty_total") + 1e-3)).cast(pl.Float32).alias("item_recent_share"),
        (pl.col("qty_total") > 0).cast(pl.Float32).alias("has_purchase_history"),
        (pl.col("event_score_total") > 0).cast(pl.Float32).alias("has_event_history"),
    ).drop("last_date", "first_date", "last_event_date")

    if target_start is not None:
        labels = (
            pl.scan_parquet(TRANSACTION_PATH)
            .filter((pl.col("updated_date") >= target_start) & (pl.col("updated_date") < target_end))
            .select("customer_id", "item_id")
            .unique()
            .with_columns(pl.lit(1, dtype=pl.Int8).alias("label"))
        )
        features = features.join(labels, on=["customer_id", "item_id"], how="left").with_columns(pl.col("label").fill_null(0))
        if sample_neg_pct is not None:
            features = (
                features
                .with_columns((pl.struct(["customer_id", "item_id"]).hash(seed=hash_seed) % 100).alias("hmod"))
                .filter((pl.col("label") == 1) | (pl.col("hmod") < sample_neg_pct))
                .drop("hmod")
            )

    df = features.collect()
    print(f"Built {df.height:,} rows x {df.width} cols in {time.time() - t0:.1f}s")
    return df


In [ ]:
def list_dict(df: pl.DataFrame, value_col: str = "items") -> dict[int, list[str]]:
    return {int(row["customer_id"]): list(row[value_col]) for row in df.iter_rows(named=True)}


def add_unique(out: list[str], seen: set[str], items: list[str] | None, limit: int = K) -> None:
    if not items:
        return
    for item_id in items:
        if item_id not in seen:
            out.append(item_id)
            seen.add(item_id)
            if len(out) >= limit:
                return


def actual_purchase_sets(start: datetime, end: datetime) -> dict[int, list[str]]:
    return list_dict(
        (
            pl.scan_parquet(TRANSACTION_PATH)
            .filter((pl.col("updated_date") >= start) & (pl.col("updated_date") < end))
            .group_by("customer_id")
            .agg(pl.col("item_id").unique().alias("items"))
            .collect()
        )
    )


def evaluate(recommendations: dict[int, list[str]], actuals: dict[int, list[str]], k: int = K) -> dict:
    n = len(actuals)
    hit_total = 0
    iou_sum = 0.0
    rr_sum = 0.0
    precision_sum = 0.0
    ap_sum = 0.0

    for customer_id, actual_items in actuals.items():
        actual_set = set(actual_items)
        pred = recommendations.get(customer_id, [])[:k]
        pred_set = set(pred)
        hits = [item_id in actual_set for item_id in pred]
        hit_count = sum(hits)
        hit_total += hit_count
        union = len(pred_set | actual_set)
        if union:
            iou_sum += len(pred_set & actual_set) / union
        for rank, hit in enumerate(hits, start=1):
            if hit:
                rr_sum += 1.0 / rank
                break
        precision_sum += hit_count / k
        denom = min(len(actual_set), k)
        if denom:
            cumulative_hits = 0
            average_precision = 0.0
            for rank, hit in enumerate(hits, start=1):
                if hit:
                    cumulative_hits += 1
                    average_precision += cumulative_hits / rank
            ap_sum += average_precision / denom

    return {
        "users": n,
        "hit_total": hit_total,
        "iou": iou_sum / n if n else 0.0,
        f"precision@{k}": precision_sum / n if n else 0.0,
        "mrr_first_hit": rr_sum / n if n else 0.0,
        f"map@{k}": ap_sum / n if n else 0.0,
    }


def train_lgbm_ranker(
    train_df: pl.DataFrame,
    *,
    feature_cols: list[str],
    random_state: int = 9,
    model_name: str = "lgbm_ranker",
) -> LGBMClassifier:
    train_pdf = train_df.select(feature_cols + ["label"]).to_pandas()
    y = train_pdf.pop("label")
    print(f"Training {model_name}: rows={len(train_pdf):,}; features={len(feature_cols)}; positive rate={float(y.mean()):.4f}")
    model = LGBMClassifier(
        n_estimators=700,
        learning_rate=0.04,
        num_leaves=63,
        subsample=0.85,
        colsample_bytree=0.90,
        min_child_samples=20,
        objective="binary",
        n_jobs=-1,
        random_state=random_state,
        verbose=-1,
    )
    t0 = time.time()
    model.fit(train_pdf, y)
    print(f"Model fit time ({model_name}): {time.time() - t0:.1f}s")
    importance = (
        pd.DataFrame({"feature": feature_cols, "importance": model.feature_importances_})
        .sort_values("importance", ascending=False)
    )
    display(importance.head(15))
    del train_pdf, y
    gc.collect()
    return model


def score_candidates(
    model: LGBMClassifier,
    candidates: pl.DataFrame,
    *,
    feature_cols: list[str],
    batch_size: int = 1_000_000,
) -> pl.DataFrame:
    scores: list[np.ndarray] = []
    t0 = time.time()
    for batch in candidates.select(feature_cols).iter_slices(batch_size):
        scores.append(model.predict_proba(batch.to_pandas())[:, 1].astype("float32"))
    score_values = np.concatenate(scores)
    print(f"Scored {len(score_values):,} candidates in {time.time() - t0:.1f}s")
    return candidates.select("customer_id", "item_id", "qty_total").with_columns(pl.Series("score", score_values))


def recent_global_top(start: datetime, end: datetime, n: int = 200) -> list[str]:
    return (
        pl.scan_parquet(TRANSACTION_PATH)
        .filter((pl.col("updated_date") >= start) & (pl.col("updated_date") < end))
        .group_by("item_id")
        .agg(pl.col("quantity").sum().alias("qty"), pl.col("customer_id").n_unique().alias("users"))
        .sort(["qty", "users"], descending=[True, True])
        .collect()["item_id"]
        .head(n)
        .to_list()
    )


def recommendations_from_scores(
    scored: pl.DataFrame,
    customer_ids: list[int] | range,
    fallback_top: list[str],
    k: int = K,
    *,
    top_n: int | None = None,
) -> dict[int, list[str]]:
    top_n = top_n or k
    top_df = (
        scored
        .sort(["customer_id", "score", "qty_total"], descending=[False, True, True])
        .group_by("customer_id", maintain_order=True)
        .agg(pl.col("item_id").head(top_n).alias("items"))
    )
    user_top = list_dict(top_df)
    del top_df
    gc.collect()

    recommendations: dict[int, list[str]] = {}
    value_cache: dict[tuple[str, ...], list[str]] = {}
    fallback = list(fallback_top[:k])
    for customer_id in customer_ids:
        out: list[str] = []
        seen: set[str] = set()
        add_unique(out, seen, user_top.get(int(customer_id)), k)
        add_unique(out, seen, fallback, k)
        key = tuple(out[:k])
        cached = value_cache.get(key)
        if cached is None:
            cached = list(key)
            value_cache[key] = cached
        recommendations[int(customer_id)] = cached

    print(f"Recommendations: {len(recommendations):,} customers; {len(value_cache):,} unique lists")
    return recommendations


## 3. Validation: train tới October, predict December

Ta train bằng target November (`Jan-Oct -> Nov`) rồi score candidates tới cuối November để đo December.

Cell bên dưới dùng cùng một feature table để so sánh ba mức:

1. `repeat_hl1_baseline`: heuristic lặp mua gần đây.
2. `lgbm_purchase_event_ranker`: LightGBM cũ không có repeat feature.
3. `lgbm_repeat_aware_ranker`: LightGBM mới có repeat feature.

Notebook assert trực tiếp `precision@10` của model mới phải cao hơn model cũ trước khi qua bước final export.


In [ ]:
validation_train = build_union_features(
    NOV_2025,
    hist_last_month=10,
    target_start=NOV_2025,
    target_end=DEC_2025,
    sample_neg_pct=20,
    hash_seed=987,
)
display(validation_train.select(pl.col("label").sum().alias("positive_rows"), pl.len().alias("sampled_rows")))

validation_base_model = train_lgbm_ranker(
    validation_train,
    feature_cols=BASE_FEATURE_COLS,
    random_state=9,
    model_name="lgbm_purchase_event_ranker",
)
validation_repeat_model = train_lgbm_ranker(
    validation_train,
    feature_cols=FEATURE_COLS,
    random_state=9,
    model_name="lgbm_repeat_aware_ranker",
)
del validation_train
gc.collect()

validation_candidates = build_union_features(DEC_2025, hist_last_month=11)
validation_actuals = actual_purchase_sets(DEC_2025, JAN_2026)
validation_customer_ids = list(validation_actuals.keys())
validation_fallback = recent_global_top(NOV_2025, DEC_2025, n=200)

metrics_rows = []

baseline_scored = validation_candidates.select("customer_id", "item_id", "qty_total", pl.col("hl1").alias("score"))
validation_recs = recommendations_from_scores(baseline_scored, validation_customer_ids, validation_fallback, K)
metrics_rows.append({"model": "repeat_hl1_baseline", **evaluate(validation_recs, validation_actuals, K)})
del baseline_scored, validation_recs
gc.collect()

validation_base_scored = score_candidates(validation_base_model, validation_candidates, feature_cols=BASE_FEATURE_COLS)
validation_recs = recommendations_from_scores(validation_base_scored, validation_customer_ids, validation_fallback, K)
metrics_rows.append({"model": "lgbm_purchase_event_ranker", **evaluate(validation_recs, validation_actuals, K)})
del validation_base_model, validation_base_scored, validation_recs
gc.collect()

validation_repeat_scored = score_candidates(validation_repeat_model, validation_candidates, feature_cols=FEATURE_COLS)
validation_recs = recommendations_from_scores(validation_repeat_scored, validation_customer_ids, validation_fallback, K)
metrics_rows.append({"model": "lgbm_repeat_aware_ranker", **evaluate(validation_recs, validation_actuals, K)})
del validation_repeat_model, validation_repeat_scored, validation_recs
gc.collect()

metrics_df = pd.DataFrame(metrics_rows)
display(metrics_df)

old_precision = float(metrics_df.loc[metrics_df["model"] == "lgbm_purchase_event_ranker", "precision@10"].iloc[0])
new_precision = float(metrics_df.loc[metrics_df["model"] == "lgbm_repeat_aware_ranker", "precision@10"].iloc[0])
absolute_gain = new_precision - old_precision
relative_gain = absolute_gain / old_precision
print(f"Precision@10 gain: {old_precision:.6f} -> {new_precision:.6f} | absolute={absolute_gain:.6f} | relative={relative_gain:.2%}")
assert new_precision > old_precision, "Repeat-aware model did not improve precision@10 on validation."

NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_SUMMARY_PATH.write_text(
    json.dumps(
        {
            "old_model": "lgbm_purchase_event_ranker",
            "new_model": "lgbm_repeat_aware_ranker",
            "old_precision@10": old_precision,
            "new_precision@10": new_precision,
            "absolute_gain": absolute_gain,
            "relative_gain": relative_gain,
            "metrics": metrics_df.to_dict(orient="records"),
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
print("Saved validation summary:", VALIDATION_SUMMARY_PATH)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
sns.barplot(data=metrics_df, x="model", y="precision@10", hue="model", legend=False, ax=axes[0], palette="Blues")
axes[0].set_title("Precision@10")
axes[0].tick_params(axis="x", rotation=20)
sns.barplot(data=metrics_df, x="model", y="map@10", hue="model", legend=False, ax=axes[1], palette="Greens")
axes[1].set_title("MAP@10")
axes[1].tick_params(axis="x", rotation=20)
plt.show()

del validation_candidates, validation_actuals
gc.collect()


## 4. Final Train And Export

Final submission được tạo theo đúng pipeline đã chọn trên validation:

1. Train `lgbm_repeat_aware_ranker` bằng nhãn `Jan-Nov -> Dec`, sau đó score candidate từ toàn bộ lịch sử 2025 cho tháng `01/2026`.
2. Xuất base repeat-aware ra `submission_all_items_repeat_features.pkl` để có checkpoint trung gian.
3. Hậu xử lý base bằng `segment fallback + brand tail`: customer cold/fallback dùng top item theo cụm `customer_id`, còn customer có lịch sử được chèn thêm 1 item theo brand mạnh nhất ở cuối top-10.

Notebook không dùng file submission cũ làm nguồn; toàn bộ `submission.pkl` được sinh lại từ dữ liệu parquet và logic bên dưới.

In [ ]:
final_train = build_union_features(
    DEC_2025,
    hist_last_month=11,
    target_start=DEC_2025,
    target_end=JAN_2026,
    sample_neg_pct=20,
    hash_seed=2026,
)
display(final_train.select(pl.col("label").sum().alias("positive_rows"), pl.len().alias("sampled_rows")))

final_model = train_lgbm_ranker(
    final_train,
    feature_cols=FEATURE_COLS,
    random_state=26,
    model_name="final_lgbm_repeat_aware_ranker",
)
del final_train
gc.collect()

final_candidates = build_union_features(JAN_2026, hist_last_month=12)
final_scored = score_candidates(final_model, final_candidates, feature_cols=FEATURE_COLS)
del final_model, final_candidates
gc.collect()

final_fallback = recent_global_top(DEC_2025, JAN_2026, n=200)
print("Base fallback top-10:", final_fallback[:10])

known_customers = (
    pl.concat(
        [
            pl.scan_parquet(TRANSACTION_PATH)
            .filter((pl.col("updated_date") >= TRAIN_START) & (pl.col("updated_date") < JAN_2026))
            .select("customer_id"),
            pl.scan_parquet(EVENT_PATH)
            .filter((pl.col("event_date") >= TRAIN_START) & (pl.col("event_date") < JAN_2026))
            .select("customer_id"),
        ]
    )
    .unique()
    .sort("customer_id")
    .collect()["customer_id"]
    .to_list()
)

observed_max_customer = max(known_customers)
fallback_max_customer = max(10_000_000, int(observed_max_customer) + 500_000)
range_customers = range(1, fallback_max_customer + 1)

print(f"Known 2025 customers: {len(known_customers):,}")
print(f"Observed max customer_id: {observed_max_customer:,}")
print(f"Fallback customer_id range: 1..{fallback_max_customer:,}")

submission = recommendations_from_scores(final_scored, range_customers, final_fallback, K)
del final_scored, known_customers
gc.collect()

NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)
with BASE_REPEAT_SUBMISSION_PATH.open("wb") as f:
    pickle.dump(submission, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved base repeat-aware submission: {BASE_REPEAT_SUBMISSION_PATH}")
print(f"Base file size: {BASE_REPEAT_SUBMISSION_PATH.stat().st_size / (1024 ** 2):,.2f} MB")

In [ ]:
def log_step(message: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {message}", flush=True)


def final_global_top(mode: str, n: int = 200) -> list[str]:
    sort_cols = ["users", "qty"] if mode == "users" else ["qty", "users"]
    return (
        pl.scan_parquet(TRANSACTION_PATH)
        .filter((pl.col("updated_date") >= DEC_2025) & (pl.col("updated_date") < JAN_2026))
        .group_by("item_id")
        .agg(pl.col("customer_id").n_unique().alias("users"), pl.col("quantity").sum().alias("qty"))
        .sort(sort_cols, descending=[True, True])
        .collect()["item_id"]
        .head(n)
        .to_list()
    )


def final_segment_fallbacks() -> dict[int, list[str]]:
    df = (
        pl.scan_parquet(TRANSACTION_PATH)
        .filter((pl.col("updated_date") >= DEC_2025) & (pl.col("updated_date") < JAN_2026))
        .with_columns((pl.col("customer_id") // SEGMENT_WIDTH).alias("segment"))
        .group_by(["segment", "item_id"])
        .agg(pl.col("customer_id").n_unique().alias("users"), pl.col("quantity").sum().alias("qty"))
        .sort(["segment", "users", "qty"], descending=[False, True, True])
        .group_by("segment", maintain_order=True)
        .agg(pl.col("item_id").head(20).alias("items"))
        .collect()
    )
    return {int(row["segment"]): list(row["items"]) for row in df.iter_rows(named=True)}


def final_brand_top1() -> dict[int, list[str]]:
    log_step("Build brand top-1 candidates")
    meta = pl.scan_parquet(ITEM_PATH).select(["item_id", "brand"])

    brand_popular_items = (
        pl.scan_parquet(TRANSACTION_PATH)
        .filter((pl.col("updated_date") >= DEC_2025) & (pl.col("updated_date") < JAN_2026))
        .join(meta, on="item_id", how="left")
        .drop_nulls("brand")
        .group_by(["brand", "item_id"])
        .agg(pl.col("customer_id").n_unique().alias("item_users"), pl.col("quantity").sum().alias("item_qty"))
        .sort(["brand", "item_users", "item_qty"], descending=[False, True, True])
        .group_by("brand", maintain_order=True)
        .agg(pl.col("item_id").head(30).alias("item_id"), pl.col("item_users").head(30).alias("item_users"))
        .explode(["item_id", "item_users"])
    )

    user_brands = (
        pl.scan_parquet(TRANSACTION_PATH)
        .filter((pl.col("updated_date") >= TRAIN_START) & (pl.col("updated_date") < JAN_2026))
        .with_columns(pl.col("updated_date").dt.month().alias("month"))
        .join(meta, on="item_id", how="left")
        .drop_nulls("brand")
        .group_by(["customer_id", "brand"])
        .agg(
            (pl.col("quantity") * (0.70 ** (12 - pl.col("month")))).sum().alias("key_score"),
            pl.col("updated_date").max().alias("last_dt"),
        )
        .sort(["customer_id", "key_score", "last_dt"], descending=[False, True, True])
        .group_by("customer_id", maintain_order=True)
        .agg(pl.col("brand").head(5).alias("brand"), pl.col("key_score").head(5).alias("key_score"))
        .explode(["brand", "key_score"])
    )

    scored = (
        user_brands.join(brand_popular_items, on="brand", how="inner")
        .group_by(["customer_id", "item_id"])
        .agg(
            (pl.col("key_score").log1p() * pl.col("item_users").log1p()).max().alias("score"),
            pl.col("item_users").max().alias("tie"),
        )
        .collect()
    )
    top = (
        scored.sort(["customer_id", "score", "tie"], descending=[False, True, True])
        .group_by("customer_id", maintain_order=True)
        .agg(pl.col("item_id").head(1).alias("items"))
    )
    out = {int(row["customer_id"]): list(row["items"]) for row in top.iter_rows(named=True)}
    del scored, top
    gc.collect()
    log_step(f"Brand candidate users: {len(out):,}")
    return out


def fill_unique_final(seq: list[str], fallback: list[str]) -> list[str]:
    out: list[str] = []
    seen: set[str] = set()
    for item_id in seq:
        if item_id not in seen:
            out.append(item_id)
            seen.add(item_id)
            if len(out) >= K:
                return out
    for item_id in fallback:
        if item_id not in seen:
            out.append(item_id)
            seen.add(item_id)
            if len(out) >= K:
                return out
    return out[:K]

In [ ]:
if "submission" not in globals():
    with BASE_REPEAT_SUBMISSION_PATH.open("rb") as f:
        submission = pickle.load(f)

old_fallback_qty = final_global_top("qty", n=20)[:K]
fallback_users = final_global_top("users", n=200)
segment_fallback = final_segment_fallbacks()
brand_top1 = final_brand_top1()

old_fallback_tuple = tuple(old_fallback_qty)
changed_cold = 0
changed_brand = 0
segment_replacement_cache: dict[int, list[str]] = {}

for customer_id, recs in submission.items():
    segment_id = int(customer_id) // SEGMENT_WIDTH
    fallback = segment_fallback.get(segment_id, fallback_users)

    if tuple(recs[:K]) == old_fallback_tuple:
        new_recs = segment_replacement_cache.get(segment_id)
        if new_recs is None:
            new_recs = fill_unique_final([], fallback)
            segment_replacement_cache[segment_id] = new_recs
        if new_recs != recs:
            submission[customer_id] = new_recs
            changed_cold += 1
        continue

    brand_items = brand_top1.get(int(customer_id), [])
    if brand_items:
        new_recs = fill_unique_final(recs[:9] + brand_items + recs[9:], fallback)
        if new_recs != recs:
            submission[customer_id] = new_recs
            changed_brand += 1

del segment_fallback, brand_top1, segment_replacement_cache
gc.collect()


tmp_output = OUTPUT_PATH.with_suffix(".pkl.tmp")
with tmp_output.open("wb") as f:
    pickle.dump(submission, f, protocol=pickle.HIGHEST_PROTOCOL)
tmp_output.replace(OUTPUT_PATH)

final_summary = {
    "method": "segment_fallback_width_500000_plus_brand_replace_tail1",
    "base_submission": str(BASE_REPEAT_SUBMISSION_PATH),
    "output": str(OUTPUT_PATH),
    "customers": len(submission),
    "changed_cold_or_fallback_rows": changed_cold,
    "changed_brand_tail_rows": changed_brand,
    "validation_reference": {
        "old_repeat_feature_precision@10": 0.08574194804382136,
        "base_lgbm_with_users_fallback_precision@10": 0.08677293680573099,
        "segment_brand_tail1_precision@10": 0.08776417855793676,
    },
}
FINAL_SEGMENT_BRAND_SUMMARY_PATH.write_text(json.dumps(final_summary, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Changed cold/fallback rows: {changed_cold:,}")
print(f"Changed brand-tail rows: {changed_brand:,}")
print(f"Saved final submission: {OUTPUT_PATH}")
print(f"Final file size: {OUTPUT_PATH.stat().st_size / (1024 ** 2):,.2f} MB")
print(f"Saved summary: {FINAL_SEGMENT_BRAND_SUMMARY_PATH}")